# 📄 DeepSeek-OCR-2 Web App
Giao diện web để upload PDF, OCR bằng **DeepSeek-OCR-2**, theo dõi tiến trình.

**Chạy tất cả cell theo thứ tự:**
1. Cài đặt → 2. Tải model → 3. Khởi động Web UI

## 1️⃣ Cài đặt & Kiểm tra GPU

In [ ]:
!nvidia-smi
import sys, torch
print(f"\nCUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

!{sys.executable} -m pip install pymupdf flask -q
!{sys.executable} -m pip install transformers==4.45.2 tokenizers>=0.20 -q

# DeepSeek-OCR-2 dependencies
![ ! -d /content/DeepSeek-OCR-2 ] && git clone https://github.com/deepseek-ai/DeepSeek-OCR-2.git /content/DeepSeek-OCR-2
!{sys.executable} -m pip install einops addict easydict -q


print("\n✅ Cài đặt hoàn tất!")

## 2️⃣ Tải Model DeepSeek-OCR-2

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'addict', 'easydict'])

import time, torch, os
from transformers import AutoModel, AutoTokenizer

model_name = "deepseek-ai/DeepSeek-OCR-2"

print("🚀 Đang tải DeepSeek-OCR-2 (lần đầu sẽ download ~6GB)...")
t0 = time.time()

deepseek_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

deepseek_model = AutoModel.from_pretrained(
    model_name,
    _attn_implementation='eager',
    trust_remote_code=True,
    use_safetensors=True
).eval().cuda().to(torch.bfloat16)

model = deepseek_model
tokenizer = deepseek_tokenizer

vram = torch.cuda.memory_allocated() / 1024**3
print(f"\n✅ DeepSeek-OCR-2 đã tải! ({time.time()-t0:.0f}s, VRAM: {vram:.1f} GB)")


## 3️⃣ Khởi động Web UI

In [ ]:
# Reference model from cell 2
import os, sys, time, glob, json, shutil, threading, traceback, gc
from pathlib import Path
from flask import Flask, request, jsonify, send_file, send_from_directory
import fitz

INPUT_DIR = "/content/input_pdfs"
OUTPUT_DIR = "/content/output_markdown"
WEB_PORT = 5000
DPI = 200

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === OCR State ===
ocr_state = {"running":False,"current_file":"","current_page":0,"total_pages":0,
             "files_done":0,"files_total":0,"results":[],"error":None,"start_time":0}
state_lock = threading.Lock()

def pdf_to_images(pdf_path):
    doc = fitz.open(pdf_path)
    imgs = []
    td = f"/tmp/pdf_pages/{Path(pdf_path).stem}"
    os.makedirs(td, exist_ok=True)
    for i in range(len(doc)):
        z = DPI/72
        pix = doc[i].get_pixmap(matrix=fitz.Matrix(z,z))
        p = os.path.join(td, f"p_{i+1:04d}.png")
        pix.save(p); imgs.append(p)
    doc.close()
    return imgs


def ocr_deepseek(imgs, on_page=None):
    """OCR bằng DeepSeek-OCR-2 via Transformers."""
    mds = []
    page_times = []
    prompt = '<image>\n<|grounding|>Convert the document to markdown.'
    for i, ip in enumerate(imgs):
        t0 = time.time()
        if on_page: on_page(i+1, len(imgs))
        out_dir = f'/tmp/deepseek_ocr_out/page_{i}'
        os.makedirs(out_dir, exist_ok=True)
        os.makedirs(f'{out_dir}/images', exist_ok=True)
        print(f"   🔍 Trang {i+1}/{len(imgs)}...")
        try:
            deepseek_model.infer(
                deepseek_tokenizer, prompt=prompt, image_file=ip,
                output_path=out_dir,
                base_size=1024, image_size=768, crop_mode=True, save_results=True
            )
        except Exception as e:
            print(f"   ❌ Lỗi trang {i+1}: {e}")
            traceback.print_exc()
            mds.append(f'[Lỗi trang {i+1}: {str(e)[:200]}]')
            page_times.append(round(time.time()-t0, 2))
            continue
        # Đọc kết quả từ file
        result_file = os.path.join(out_dir, 'result.mmd')
        if os.path.exists(result_file):
            with open(result_file, 'r', encoding='utf-8') as f:
                text = f.read().strip()
            print(f"   ✅ Trang {i+1} OK ({len(text)} chars)")
        else:
            found = [f for f in os.listdir(out_dir) if f.endswith(('.mmd','.md','.txt'))]
            if found:
                with open(os.path.join(out_dir, found[0]), 'r', encoding='utf-8') as f:
                    text = f.read().strip()
            else:
                text = f'[Không có kết quả. Files trong output: {os.listdir(out_dir)}]'
                print(f"   ⚠️ Trang {i+1}: không tìm thấy file kết quả")
        mds.append(text)
        page_times.append(round(time.time()-t0, 2))
    shutil.rmtree('/tmp/deepseek_ocr_out', ignore_errors=True)
    return "\n\n---\n\n".join(mds), page_times


def ocr_worker(pdfs):
    global ocr_state
    with state_lock:
        ocr_state.update({"running":True,"files_done":0,"files_total":len(pdfs),
                         "results":[],"error":None,"start_time":time.time()})
    for idx, pdf in enumerate(pdfs):
        nm = Path(pdf).stem
        with state_lock:
            ocr_state["current_file"]=Path(pdf).name; ocr_state["current_page"]=0; ocr_state["total_pages"]=0
        try:
            t0 = time.time()
            imgs = pdf_to_images(pdf)
            t_img = time.time() - t0
            with state_lock: ocr_state["total_pages"]=len(imgs)
            def on_pg(p,t):
                with state_lock: ocr_state["current_page"]=p
            t1 = time.time()
            md, page_times = ocr_deepseek(imgs, on_page=on_pg)
            t_ocr = time.time() - t1
            out = os.path.join(OUTPUT_DIR, f"{nm}.md")
            with open(out,'w',encoding='utf-8') as f:
                f.write(f"<!-- OCR by DeepSeek-OCR-2 | {Path(pdf).name} -->\n\n{md}")
            with state_lock:
                ocr_state["files_done"]=idx+1
                ocr_state["results"].append({
                    "file":Path(pdf).name,"pages":len(imgs),"output":f"{nm}.md","status":"ok",
                    "time_ocr":round(t_ocr,2),"time_img":round(t_img,2),
                    "avg_per_page":round(t_ocr/max(len(imgs),1),2)
                })
            shutil.rmtree(f"/tmp/pdf_pages/{nm}", ignore_errors=True)
        except Exception as e:
            print(f"❌ Error: {pdf} — {e}")
            traceback.print_exc()
            with state_lock:
                ocr_state["files_done"]=idx+1
                ocr_state["results"].append({"file":Path(pdf).name,"pages":0,"output":"","status":f"error:{e}",
                    "time_ocr":0,"time_img":0,"avg_per_page":0})
    with state_lock: ocr_state["running"]=False; ocr_state["current_file"]=""


# === Flask App ===
app = Flask(__name__)

@app.route("/")
def index(): return send_file("/content/deepseek_ui.html")

@app.route("/api/upload", methods=["POST"])
def upload():
    ups = []
    for f in request.files.getlist("files"):
        if f.filename and f.filename.lower().endswith(".pdf"):
            dest = os.path.join(INPUT_DIR, f.filename)
            f.save(dest)
            ups.append({"name":f.filename,"size":os.path.getsize(dest)})
    return jsonify({"uploaded":ups})

@app.route("/api/files")
def list_files():
    pdfs = [{"name":f,"size":os.path.getsize(os.path.join(INPUT_DIR,f))} for f in sorted(os.listdir(INPUT_DIR)) if f.lower().endswith(".pdf")]
    mds = [{"name":f,"size":os.path.getsize(os.path.join(OUTPUT_DIR,f))} for f in sorted(os.listdir(OUTPUT_DIR)) if f.lower().endswith(".md")]
    return jsonify({"pdfs":pdfs,"markdowns":mds})

@app.route("/api/ocr", methods=["POST"])
def start_ocr():
    if ocr_state["running"]: return jsonify({"error":"OCR đang chạy"}), 409
    pdfs = sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdf")))
    if not pdfs: return jsonify({"error":"Không có file PDF"}), 400
    threading.Thread(target=ocr_worker, args=(pdfs,), daemon=True).start()
    return jsonify({"started":len(pdfs)})

@app.route("/api/status")
def status():
    with state_lock:
        el = time.time()-ocr_state["start_time"] if ocr_state["start_time"] else 0
        return jsonify({**ocr_state, "elapsed":round(el,1)})

@app.route("/api/preview/<fn>")
def preview(fn):
    p = os.path.join(OUTPUT_DIR, fn)
    if not os.path.exists(p): return jsonify({"error":"Not found"}), 404
    return jsonify({"content":open(p,'r',encoding='utf-8').read(),"name":fn})

@app.route("/api/download/<fn>")
def download(fn): return send_from_directory(OUTPUT_DIR, fn, as_attachment=True)

@app.route("/api/download_all")
def download_all():
    shutil.make_archive("/tmp/ocr_results","zip",OUTPUT_DIR)
    return send_file("/tmp/ocr_results.zip", as_attachment=True, download_name="ocr_results.zip")

@app.route("/api/delete/<fn>", methods=["DELETE"])
def delete_file(fn):
    for d in [INPUT_DIR, OUTPUT_DIR]:
        p = os.path.join(d, fn)
        if os.path.exists(p): os.remove(p)
    return jsonify({"deleted":fn})

print("✅ Flask app sẵn sàng")

In [ ]:
%%writefile /content/deepseek_ui.html
<!DOCTYPE html>
<html lang="vi">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>DeepSeek-OCR-2 — PDF to Markdown</title>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">
<style>
:root{--bg:#0a0e17;--bg2:#111827;--card:rgba(17,24,39,0.7);--border:rgba(255,255,255,0.08);--text:#e5e7eb;--text2:#9ca3af;--accent:#06b6d4;--accent2:#22d3ee;--glow:rgba(34,211,238,0.3);--ok:#10b981;--err:#ef4444}
*{margin:0;padding:0;box-sizing:border-box}
body{font-family:'Inter',system-ui,sans-serif;background:var(--bg);color:var(--text);min-height:100vh}
body::before{content:'';position:fixed;inset:0;background:radial-gradient(circle at 30% 20%,rgba(34,211,238,0.08),transparent 50%),radial-gradient(circle at 70% 80%,rgba(16,185,129,0.06),transparent 50%);z-index:0}
.ct{max-width:1100px;margin:0 auto;padding:24px 20px;position:relative;z-index:1}
.hdr{text-align:center;margin-bottom:28px;padding:24px;background:var(--card);border:1px solid var(--border);border-radius:20px;backdrop-filter:blur(20px)}
.hdr h1{font-size:26px;font-weight:700;background:linear-gradient(135deg,#22d3ee,#06b6d4,#10b981);-webkit-background-clip:text;-webkit-text-fill-color:transparent}
.hdr p{color:var(--text2);margin-top:4px;font-size:13px}
.cd{background:var(--card);border:1px solid var(--border);border-radius:16px;padding:20px;margin-bottom:16px;backdrop-filter:blur(20px)}
.cd h2{font-size:15px;font-weight:600;margin-bottom:14px;display:flex;align-items:center;gap:8px}
.uz{border:2px dashed rgba(34,211,238,0.3);border-radius:12px;padding:36px;text-align:center;cursor:pointer;transition:all .3s}
.uz:hover,.uz.dg{border-color:var(--accent);background:rgba(34,211,238,0.05);box-shadow:0 0 30px var(--glow)}
.uz .em{font-size:36px;margin-bottom:8px}
.uz p{color:var(--text2);font-size:13px}
.uz input{display:none}
.gr{display:grid;grid-template-columns:1fr 1fr;gap:16px}
@media(max-width:768px){.gr{grid-template-columns:1fr}}
.fl{max-height:240px;overflow-y:auto}
.fi{display:flex;align-items:center;justify-content:space-between;padding:9px 12px;border-radius:8px;margin-bottom:5px;background:rgba(255,255,255,0.03);border:1px solid var(--border);font-size:12px}
.fi:hover{background:rgba(255,255,255,0.06)}
.fn{flex:1;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.fs{color:var(--text2);font-size:11px;margin-left:8px}
.fa{display:flex;gap:4px;margin-left:8px}
.btn{display:inline-flex;align-items:center;gap:5px;padding:7px 14px;border-radius:8px;border:none;font-family:inherit;font-size:12px;font-weight:500;cursor:pointer;transition:all .2s;color:#fff}
.bp{background:linear-gradient(135deg,var(--accent),#0891b2);box-shadow:0 4px 15px var(--glow)}
.bp:hover{transform:translateY(-1px)}
.bp:disabled{opacity:.4;cursor:not-allowed;transform:none}
.bs{padding:4px 9px;font-size:11px;border-radius:6px}
.bg{background:rgba(255,255,255,0.06);color:var(--text2)}
.bg:hover{background:rgba(255,255,255,0.1);color:var(--text)}
.bd{background:rgba(239,68,68,0.15);color:var(--err)}
.bk{background:rgba(16,185,129,0.15);color:var(--ok)}
.bf{width:100%;justify-content:center;padding:11px;font-size:14px}
.pb{display:none}.pb.on{display:block}
.pbw{height:8px;background:rgba(255,255,255,0.06);border-radius:4px;overflow:hidden;margin:10px 0}
.pbb{height:100%;border-radius:4px;transition:width .5s;background:linear-gradient(90deg,var(--accent),var(--ok));box-shadow:0 0 12px var(--glow)}
.pi{display:flex;justify-content:space-between;font-size:11px;color:var(--text2)}
.pf{font-size:12px;color:var(--accent2);margin-bottom:6px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.sr{display:flex;gap:12px;margin-top:10px;flex-wrap:wrap}
.st{padding:6px 12px;border-radius:8px;background:rgba(255,255,255,0.03);border:1px solid var(--border);font-size:11px}
.st b{color:var(--accent2)}
.mo{display:none;position:fixed;inset:0;background:rgba(0,0,0,0.7);z-index:1000;backdrop-filter:blur(4px);justify-content:center;align-items:center;padding:20px}
.mo.on{display:flex}
.ml{background:var(--bg2);border:1px solid var(--border);border-radius:16px;width:100%;max-width:800px;max-height:85vh;display:flex;flex-direction:column}
.mh{display:flex;justify-content:space-between;align-items:center;padding:14px 18px;border-bottom:1px solid var(--border)}
.mc{background:none;border:none;color:var(--text2);font-size:18px;cursor:pointer;padding:4px 8px;border-radius:6px}
.mc:hover{background:rgba(255,255,255,0.1)}
.mb{padding:18px;overflow-y:auto;flex:1;font-size:13px;line-height:1.7;white-space:pre-wrap;word-wrap:break-word}
.emp{text-align:center;padding:20px;color:var(--text2);font-size:12px}
::-webkit-scrollbar{width:5px}::-webkit-scrollbar-thumb{background:rgba(255,255,255,0.1);border-radius:3px}
.tt{position:fixed;bottom:20px;right:20px;z-index:2000;padding:10px 18px;border-radius:10px;font-size:12px;background:var(--bg2);border:1px solid var(--border);box-shadow:0 8px 32px rgba(0,0,0,.4);transform:translateY(80px);opacity:0;transition:all .3s}
.tt.on{transform:translateY(0);opacity:1}
@keyframes pl{0%,100%{opacity:1}50%{opacity:.5}}
.pls{animation:pl 1.5s ease-in-out infinite}
.tr{margin-top:16px}.tr table{width:100%;border-collapse:collapse;font-size:11px}.tr th{text-align:left;padding:8px 10px;border-bottom:2px solid var(--border);color:var(--text2);font-weight:600}.tr td{padding:7px 10px;border-bottom:1px solid var(--border)}.tr tr:hover td{background:rgba(255,255,255,0.03)}.tr tfoot td{border-top:2px solid var(--border);font-weight:600}
</style>
</head>
<body>
<div class="ct">
  <div class="hdr"><h1>🟢 DeepSeek-OCR-2 — PDF to Markdown</h1><p>Upload PDF → OCR tự động → Tải Markdown</p></div>
  <div class="cd">
    <h2>📤 Upload PDF</h2>
    <div class="uz" id="uz"><div class="em">📁</div><p>Kéo thả file PDF hoặc <b>nhấn để chọn</b></p><input type="file" id="fi" multiple accept=".pdf"></div>
  </div>
  <div class="gr">
    <div class="cd">
      <h2>📋 File PDF <span id="pc" style="color:var(--text2);font-weight:400"></span></h2>
      <div id="pl" class="fl"><div class="emp">Chưa có file</div></div>
      <div style="margin-top:12px"><button class="btn bp bf" id="bo" onclick="sOcr()">🔍 Bắt đầu OCR</button></div>
    </div>
    <div class="cd">
      <h2>✅ Kết quả <span id="mc2" style="color:var(--text2);font-weight:400"></span></h2>
      <div id="ml" class="fl"><div class="emp">Chưa có kết quả</div></div>
      <div style="margin-top:12px"><button class="btn bk bf" onclick="dAll()">📦 Tải tất cả (ZIP)</button></div>
    </div>
  </div>
  <div class="cd pb" id="pbx">
    <h2><span class="pls">⚡</span> Đang OCR...</h2>
    <div class="pf" id="pfl">—</div>
    <div class="pbw"><div class="pbb" id="pba" style="width:0%"></div></div>
    <div class="pi"><span id="ptx">0%</span><span id="ptm">0s</span></div>
    <div class="sr"><div class="st">File: <b id="sf">0/0</b></div><div class="st">Trang: <b id="sp">0/0</b></div><div class="st">⏱ <b id="se">0s</b></div></div>
  </div>
  <div class="cd pb" id="trx"><h2>📊 Thời gian xử lý</h2><div id="trbody"></div></div>
</div>
<div class="mo" id="mov" onclick="if(event.target===this)cMo()"><div class="ml"><div class="mh"><h3 id="mt" style="font-size:14px">Preview</h3><button class="mc" onclick="cMo()">✕</button></div><div class="mb" id="mbd"></div></div></div>
<div class="tt" id="tt"></div>
<script>
let pt=null;
function fm(b){return b<1024?b+'B':b<1048576?(b/1024).toFixed(1)+'KB':(b/1048576).toFixed(1)+'MB'}
function tw(m){const e=document.getElementById('tt');e.textContent=m;e.className='tt on';setTimeout(()=>e.classList.remove('on'),3000)}
const uz=document.getElementById('uz'),fi=document.getElementById('fi');
uz.onclick=()=>fi.click();
uz.ondragover=e=>{e.preventDefault();uz.classList.add('dg')};
uz.ondragleave=()=>uz.classList.remove('dg');
uz.ondrop=e=>{e.preventDefault();uz.classList.remove('dg');uF(e.dataTransfer.files)};
fi.onchange=()=>{if(fi.files.length)uF(fi.files)};
async function uF(fs){const fd=new FormData();let c=0;for(const f of fs)if(f.name.toLowerCase().endsWith('.pdf')){fd.append('files',f);c++}if(!c){tw('Chỉ hỗ trợ PDF');return}try{const r=await(await fetch('/api/upload',{method:'POST',body:fd})).json();tw('Upload '+r.uploaded.length+' file');rF()}catch(e){tw('Lỗi: '+e.message)}}
async function rF(){try{const d=await(await fetch('/api/files')).json();rP(d.pdfs);rM(d.markdowns)}catch(e){console.error(e)}}
function rP(fs){const e=document.getElementById('pl');document.getElementById('pc').textContent=fs.length?'('+fs.length+')':'';if(!fs.length){e.innerHTML='<div class="emp">Chưa có file</div>';return}e.innerHTML=fs.map(f=>'<div class="fi"><span class="fn">📄 '+f.name+'</span><span class="fs">'+fm(f.size)+'</span><div class="fa"><button class="btn bs bd" onclick="dF(\''+f.name+'\')">&times;</button></div></div>').join('')}
function rM(fs){const e=document.getElementById('ml');document.getElementById('mc2').textContent=fs.length?'('+fs.length+')':'';if(!fs.length){e.innerHTML='<div class="emp">Chưa có kết quả</div>';return}e.innerHTML=fs.map(f=>'<div class="fi"><span class="fn">📝 '+f.name+'</span><span class="fs">'+fm(f.size)+'</span><div class="fa"><button class="btn bs bg" onclick="pV(\''+f.name+'\')">&equiv;</button><button class="btn bs bk" onclick="dD(\''+f.name+'\')">&darr;</button></div></div>').join('')}
async function sOcr(){try{const r=await(await fetch('/api/ocr',{method:'POST',headers:{'Content-Type':'application/json'}})).json();if(r.error){tw(r.error);return}tw('OCR '+r.started+' file');document.getElementById('pbx').classList.add('on');document.getElementById('bo').disabled=true;sP()}catch(e){tw('Lỗi: '+e.message)}}
function sP(){if(pt)clearInterval(pt);pt=setInterval(pS,1000)}
async function pS(){try{const s=await(await fetch('/api/status')).json();const p=s.files_total>0?Math.round(((s.files_done+(s.total_pages>0?s.current_page/s.total_pages:0))/s.files_total)*100):0;document.getElementById('pba').style.width=p+'%';document.getElementById('ptx').textContent=p+'%';document.getElementById('pfl').textContent=s.current_file||'—';document.getElementById('ptm').textContent=s.elapsed+'s';document.getElementById('sf').textContent=s.files_done+'/'+s.files_total;document.getElementById('sp').textContent=s.current_page+'/'+s.total_pages;document.getElementById('se').textContent=s.elapsed+'s';if(!s.running&&s.files_total>0){clearInterval(pt);pt=null;document.getElementById('pba').style.width='100%';document.getElementById('ptx').textContent='100%';document.getElementById('bo').disabled=false;const ok=s.results.filter(r=>r.status==='ok').length;tw('Xong! '+ok+'/'+s.files_total+' OK ('+s.elapsed+'s)');buildReport(s);setTimeout(()=>{document.getElementById('pbx').classList.remove('on');rF()},2000)}}catch(e){console.error(e)}}
function buildReport(s){if(!s.results||!s.results.length)return;const box=document.getElementById('trx');box.classList.add('on');let h='<div class="tr"><table><thead><tr><th>File</th><th>Trang</th><th>PDF→Ảnh</th><th>OCR</th><th>TB/trang</th></tr></thead><tbody>';let tP=0,tO=0,tI=0;s.results.filter(r=>r.status==='ok').forEach(r=>{h+='<tr><td>'+r.file+'</td><td>'+r.pages+'</td><td>'+r.time_img+'s</td><td>'+r.time_ocr+'s</td><td>'+r.avg_per_page+'s</td></tr>';tP+=r.pages;tO+=r.time_ocr;tI+=r.time_img});h+='</tbody><tfoot><tr><td><b>TỔNG</b></td><td>'+tP+'</td><td>'+tI.toFixed(1)+'s</td><td>'+tO.toFixed(1)+'s</td><td>'+(tO/Math.max(tP,1)).toFixed(2)+'s</td></tr></tfoot></table></div>';document.getElementById('trbody').innerHTML=h}
async function dF(n){await fetch('/api/delete/'+n,{method:'DELETE'});rF()}
async function pV(n){try{const d=await(await fetch('/api/preview/'+n)).json();document.getElementById('mt').textContent=n;document.getElementById('mbd').textContent=d.content;document.getElementById('mov').classList.add('on')}catch(e){tw('Lỗi')}}
function cMo(){document.getElementById('mov').classList.remove('on')}
function dD(n){window.open('/api/download/'+n)}
function dAll(){window.open('/api/download_all')}
rF();
fetch('/api/status').then(r=>r.json()).then(s=>{if(s.running){document.getElementById('pbx').classList.add('on');document.getElementById('bo').disabled=true;sP()}});
</script>
</body></html>

In [ ]:
import threading
from google.colab import output

server_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=WEB_PORT, debug=False, use_reloader=False),
    daemon=True
)
server_thread.start()

import time
time.sleep(2)

print(f"✅ Web server đang chạy trên port {WEB_PORT}")
print("\n🌐 Mở giao diện web bên dưới:")

output.serve_kernel_port_as_iframe(WEB_PORT, height=800)